# Create HAL config and shutter files (cells + transit only)

This `merfish_multi_z` variant runs a **full-depth DAPI (cells) round first**,
measures each FOV's real tissue depth from it (notebook 04), then generates
per-tier bits hal_configs/shutters afterward (notebook 05) — so this notebook
only builds the **cells** and **transit** configs. The bits config used to
live here too; for this variant it's deliberately deferred until the tissue-
thickness measurement exists, since the whole point is to size the bits
z-range per FOV instead of guessing a single fixed depth upfront.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs import (
    get_frame_table, get_transit_frame_table, get_color_sequence_name,
    create_shutter_file, create_hal_config, power_dict_to_channel_list,
    frame_table_filename, shutter_filename, hal_config_filename,
)
from MERci.acquisition.display import print_frame_table, display_xml
from MERci.visualization       import visualize_shutter_sequence

In [ ]:
SETTINGS_DIR = SAMPLE_DIR / "settings"
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

MICROSCOPE = "ST2"

# ── Auto-detect HAL config template ────────────────────────────────────
# Looks for hal-config-*{MICROSCOPE}*.xml in MERci/data/configs/hal/
# (case-insensitive match on the microscope name)
_hal_dir        = MERCI_DIR / "data" / "configs" / "hal"
_hal_candidates = sorted(
    p for p in _hal_dir.glob("hal-config-*.xml")
    if MICROSCOPE.lower() in p.name.lower()
)
if not _hal_candidates:
    raise FileNotFoundError(
        f"No HAL template found for microscope '{MICROSCOPE}' in {_hal_dir}"
    )
HAL_TEMPLATE = _hal_candidates[0]

# ── Imaging file format and camera settings ────────────────────────────
# FILE_TYPE options: ".zarr" (default), ".dax", ".tiff"
FILE_TYPE     = ".zarr"
EXPOSURE_TIME = 0.15      # seconds

# ── Per-channel laser power ─────────────────────────────────────────────
# POWER maps each excitation wavelength (nm) to its laser power (0..1). It is
# written ONLY into the HAL config's <default_power> list, ordered per channel
# via the microscope colour->channel map (power_dict_to_channel_list) -- this
# is the actual acquisition power. Every shutter <event>'s own <power> is
# always POWER_DEFAULT (1.0) regardless of colour: it is a full-modulation
# flag relative to <default_power>, not an independent absolute power, so
# writing the same real per-colour intensity into both places double-applies
# the scaling on real hardware (a bug this notebook had for a while -- see
# create_shutter_file's docstring).
# Any wavelength not listed falls back to POWER_DEFAULT.
# NB: only list wavelengths the microscope actually has. ST2/MFX are 4-channel
# (650/560/488/405, no 750) — listing 750 here raises a ValueError.
POWER = {
    650: 1.00,
    560: 1.00,
    488: 1.00,
    405: 1.00,
}
POWER_DEFAULT = 1.0

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SETTINGS_DIR : {SETTINGS_DIR}")
print(f"METADATA_DIR : {METADATA_DIR}")
print(f"HAL template : {HAL_TEMPLATE.name}")
print(f"Power (nm->power) : {POWER}")
print(f"HAL default_power : {power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT)}")

## create hal config for cells

Full depth -- this is the calibration round notebook 04 (measure_tissue_thickness)
will measure real tissue depth from, so its z-range should comfortably cover
the deepest tissue you expect anywhere in the sample.

In [ ]:
# ── Define imaging sequence ────────────────────────────────────────────
z_bead    = 0
z_min     = 0.5
z_max     = 70
z_step    = 0.5
z_pos     = np.arange(z_min, z_max+1, z_step)
bead_seq  = [488]
color_seq = [405, 488]
end_seq   = [488]

# Scan mode (see bits block for description; typically "interleaved" for a single-color round)
SCAN_MODE = "interleaved"

# z return mode — how the objective travels back to z_bead after the stack:
#   "progressive" : blank frames step the objective down to z_bead in increments of RETURN_STEP
#   "instant"     : single jump back to z_bead (no intermediate frames)
Z_RETURN_MODE = "progressive"
RETURN_STEP   = 10


frame_table = get_frame_table(z_bead, bead_seq, color_seq, end_seq, z_pos,
                               microscope=MICROSCOPE, scan_mode=SCAN_MODE,
                               z_return_mode=Z_RETURN_MODE, return_step=RETURN_STEP)
name        = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)
print(f"Color sequence name: {name}")
print_frame_table(frame_table)

# ── Save frame table ────────────────────────────────────────────────────
ft_path = METADATA_DIR / frame_table_filename("cells", name)
frame_table.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")

# ── Write shutter file (always full power -- POWER only sets HAL <default_power> below) ──
shutter_name = shutter_filename("cells", name)
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(frame_table, shutter_path, default_power=POWER_DEFAULT)
print(f"Shutter file saved: {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config (per-channel <default_power> from POWER) ────────────
hal_name   = hal_config_filename(MICROSCOPE, "cells", name)
hal_output = SETTINGS_DIR / hal_name
create_hal_config(HAL_TEMPLATE, frame_table, shutter_name, hal_output,
                  default_power=power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT),
                  file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)
print(f"HAL config saved: {hal_output}")
display_xml(hal_output)

# ── Visualise shutter sequence ──────────────────────────────────────────
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)

## create hal config for transit

Transit FOVs sit between two tissue boundaries and are visited only to move the
stage smoothly between sections (see notebook 03). No data is collected there:
each transit FOV is imaged as `N_TRANSIT_BLANK` blank (laser-off) frames held at
the bead focus. This produces a dedicated `hal-config-{mic}-transit-*.xml` +
shutter that notebook 06 assigns to the transit movies.

In [ ]:
# ── Define transit acquisition ─────────────────────────────────────────
z_bead          = 0     # bead focus; transit frames stay here
N_TRANSIT_BLANK = 2     # blank frames per transit FOV

transit_ft   = get_transit_frame_table(bead_z=z_bead, n_blank=N_TRANSIT_BLANK)
transit_name = get_color_sequence_name(transit_ft)   # e.g. "blkf2"
print(f"Transit sequence name: {transit_name}  ({N_TRANSIT_BLANK} blank frames)")
print_frame_table(transit_ft)

# ── Save frame table ────────────────────────────────────────────────────
ft_path = METADATA_DIR / frame_table_filename("transit", transit_name)
transit_ft.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")

# ── Write shutter file (all frames blank -> no events) ───────────────────
shutter_name = shutter_filename("transit", transit_name)
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(transit_ft, shutter_path)
print(f"Shutter file saved: {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config ────────────────────────────────────────────────────
hal_name   = hal_config_filename(MICROSCOPE, "transit", transit_name)
hal_output = SETTINGS_DIR / hal_name
create_hal_config(HAL_TEMPLATE, transit_ft, shutter_name, hal_output,
                  default_power=power_dict_to_channel_list(POWER, MICROSCOPE, POWER_DEFAULT),
                  file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)
print(f"HAL config saved: {hal_output}")
display_xml(hal_output)